# GATU Export and Compare

This notebook writes all generated artifacts to `outputs/`.

## Purpose

Prepare representative ViraLift failure cases for GATU and compare GATU-exported annotations back against ViraLift. GATU execution itself is external/manual.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.validation._shared.validation_utils import *

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
SOURCE_CASES = ROOT / 'app/validation/04_tblastn_truth_breakdown_100seq/outputs/tblastn_review_cases.tsv'
GATU_OUTPUT_DIR = OUTPUT_DIR / 'gatu_output'
EXPORT_DIR = OUTPUT_DIR / 'gatu_inputs'
EXPORT_DIR.mkdir(exist_ok=True)
GATU_OUTPUT_DIR.mkdir(exist_ok=True)

if SOURCE_CASES.exists():
    cases = pd.read_csv(SOURCE_CASES, sep='\t')
else:
    cases = pd.DataFrame()

cases.head()

,virus,record_id,method,pred_name,source_name,pred_start,pred_end,strand,status,coverage,...,truth_name,truth_start,truth_end,best_overlap_name,best_iou,iou,name_match,coord_correct,exact_match,failure_mode
0,FMD,AY687333.1,tblastn,VP1,NaN,3251.0,3880.0,+,ok,1.0000,...,VP1,3251.0,3886.0,VP1,0.9906,0.9906,True,True,False,Boundary offset
1,FMD,AY687334.1,tblastn,VP1,NaN,3250.0,3876.0,+,ok,1.0000,...,VP1,3250.0,3882.0,VP1,0.9905,0.9905,True,True,False,Boundary offset
2,FMD,DQ533483.2,tblastn,2A,NaN,3893.0,3943.0,+,ok,0.9444,...,2A,3890.0,3943.0,2A,0.9444,0.9444,True,True,False,Boundary offset
3,FMD,DQ989304.1,tblastn,2A,NaN,3934.0,3984.0,+,ok,0.9444,...,2A,3931.0,3984.0,2A,0.9444,0.9444,True,True,False,Boundary offset
4,FMD,DQ989313.1,tblastn,2A,NaN,3980.0,4030.0,+,ok,0.9444,...,2A,3977.0,4030.0,2A,0.9444,0.9444,True,True,False,Boundary offset


In [3]:
# Pick a small, reviewable set: top 2 records per virus / failure mode / predicted gene.
if not cases.empty:
    selected = (
        cases[cases['failure_mode'] != 'Correct']
        .groupby(['virus', 'failure_mode', 'pred_name'], dropna=False)
        .head(2)
        .reset_index(drop=True)
    )
    selected.to_csv(OUTPUT_DIR / 'gatu_manifest.csv', index=False)
else:
    selected = pd.DataFrame()
selected.head(30)

,virus,record_id,method,pred_name,source_name,pred_start,pred_end,strand,status,coverage,...,truth_name,truth_start,truth_end,best_overlap_name,best_iou,iou,name_match,coord_correct,exact_match,failure_mode
0,FMD,AY687333.1,tblastn,VP1,NaN,3251.0,3880.0,+,ok,1.0000,...,VP1,3251.0,3886.0,VP1,0.9906,0.9906,True,True,False,Boundary offset
1,FMD,AY687334.1,tblastn,VP1,NaN,3250.0,3876.0,+,ok,1.0000,...,VP1,3250.0,3882.0,VP1,0.9905,0.9905,True,True,False,Boundary offset
2,FMD,DQ533483.2,tblastn,2A,NaN,3893.0,3943.0,+,ok,0.9444,...,2A,3890.0,3943.0,2A,0.9444,0.9444,True,True,False,Boundary offset
3,FMD,DQ989304.1,tblastn,2A,NaN,3934.0,3984.0,+,ok,0.9444,...,2A,3931.0,3984.0,2A,0.9444,0.9444,True,True,False,Boundary offset
4,FMD,DQ989317.1,tblastn,Lpro,NaN,1194.0,1784.0,+,ok,0.9801,...,Lpro,1182.0,1784.0,Lpro,0.9801,0.9801,True,True,False,Boundary offset
5,FMD,DQ989318.1,tblastn,Lpro,NaN,1190.0,1780.0,+,ok,0.9801,...,Lpro,1178.0,1780.0,Lpro,0.9801,0.9801,True,True,False,Boundary offset
6,FMD,PV805282.1,tblastn,3Cpro,NaN,6063.0,6689.0,+,ok,0.9812,...,3Cpro,6051.0,6689.0,3Cpro,0.9812,0.9812,True,True,False,Boundary offset
7,FMD,PV805283.1,tblastn,3B,NaN,5835.0,6044.0,+,ok,0.9859,...,3B,5835.0,6047.0,3B,0.9859,0.9859,True,True,False,Boundary offset
8,FMD,PV805283.1,tblastn,3Cpro,NaN,6060.0,6686.0,+,ok,0.9812,...,3Cpro,6048.0,6686.0,3Cpro,0.9812,0.9812,True,True,False,Boundary offset
9,FMD,DQ989316.1,tblastn,VP1,NaN,3510.0,3977.0,+,ok,0.7488,...,NaN,NaN,NaN,NaN,0.0000,0.0000,False,False,False,Not in truth


In [4]:
# Export query FASTA files for selected records.
from Bio import SeqIO

QUERY_BY_VIRUS = {
    'PRRS': CROSS_CHECK / 'PRRS_100seq_anno.gb',
    'FMD': CROSS_CHECK / 'FMD_100seq_anno.gb',
}

if not selected.empty:
    for virus, query_path in QUERY_BY_VIRUS.items():
        wanted = set(selected[selected['virus'] == virus]['record_id'])
        if not wanted:
            continue
        for record in load_genbank_records(query_path):
            if record.id in wanted:
                out = EXPORT_DIR / f'{virus}_{record.id}.fasta'
                SeqIO.write(record, out, 'fasta')
print(f'FASTA export dir: {EXPORT_DIR}')
print(f'Place GATU GenBank outputs in: {GATU_OUTPUT_DIR}')

FASTA export dir: outputs/gatu_inputs
Place GATU GenBank outputs in: outputs/gatu_output


In [5]:
# After running GATU externally, place GenBank outputs in outputs/gatu_output/.
# This cell parses those outputs and compares them with the selected ViraLift rows.

gatu_rows = []
for gb_path in GATU_OUTPUT_DIR.glob('*.gb*'):
    for record in load_genbank_records(gb_path):
        for feature in parse_cds_features(record) + parse_mat_peptides(record):
            gatu_rows.append({
                'record_id': record.id,
                'gatu_file': gb_path.name,
                'gene': feature['name'],
                'gatu_start': feature['start'],
                'gatu_end': feature['end'],
            })

gatu_df = pd.DataFrame(gatu_rows)
if not gatu_df.empty and not selected.empty:
    compare = selected.merge(gatu_df, left_on=['record_id', 'pred_name'], right_on=['record_id', 'gene'], how='outer')
    compare['gatu_iou_vs_vl'] = compare.apply(lambda r: iou(r.get('pred_start'), r.get('pred_end'), r.get('gatu_start'), r.get('gatu_end')), axis=1)
    compare.to_csv(OUTPUT_DIR / 'gatu_vs_viralift.tsv', sep='\t', index=False)
else:
    compare = pd.DataFrame()
compare.head()

""
